# Experiment 1 — Baseline ML Model Performance

Trains and evaluates all baseline models: Logistic Regression, Random Forest,
XGBoost, LightGBM, CatBoost, and TabNet across all cancer types and prediction horizons.

**Key outputs:**
- AUROC / AUPRC / F1 / Brier score per model
- ROC and PR curves overlaid
- Bootstrap 95% CIs
- Model-specific feature importances

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, precision_recall_curve

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.models.tabnet_model import TabNetTrainer
from src.evaluation.metrics import calculate_clinical_metrics
from src.evaluation.statistical_tests import bootstrap_auroc_ci

seed_everything(42)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../results/models', exist_ok=True)
print('Environment ready.')

In [ ]:
# Load 12-month prediction horizon dataset
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'results': '../results',
              'models': '../results/models', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','rbc','hemoglobin','hematocrit','mcv','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine','sodium','potassium'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)
print(f'Train: {X_train_p.shape} | Val: {X_val_p.shape} | Test: {X_test_p.shape}')

In [ ]:
# Train all baseline models
trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)

# TabNet
tabnet = TabNetTrainer(cfg)
results['tabnet'] = tabnet.train(X_train_p, y_train, X_val_p, y_val)

# Collect test AUROC for each model
drop_cols = ['subject_id', 'cancer_type', 'gender', 'age']
Xte = X_test_p.drop(columns=[c for c in drop_cols if c in X_test_p.columns]).fillna(0)

rows = []
for name, res in results.items():
    model = res['model']
    if hasattr(model, 'predict_proba'):
        prob = model.predict_proba(Xte)[:, 1]
    else:
        prob = model.predict(Xte)
    m = calculate_clinical_metrics(y_test, prob)
    m['model'] = name
    rows.append(m)

leaderboard = pd.DataFrame(rows).set_index('model').sort_values('auroc', ascending=False)
print(leaderboard[['auroc','auprc','f1_score','brier_score']].round(4))

In [ ]:
# ROC curves for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for name, res in results.items():
    model = res['model']
    prob = model.predict_proba(Xte)[:, 1] if hasattr(model, 'predict_proba') else model.predict(Xte)
    fpr, tpr, _ = roc_curve(y_test, prob)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})', lw=1.8)
    
    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc = auc(recall, precision)
    axes[1].plot(recall, precision, label=f'{name} (AUPRC={pr_auc:.3f})', lw=1.8)

axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set(title='ROC Curves — All Baselines', xlabel='FPR', ylabel='TPR')
axes[0].legend(fontsize=9)

axes[1].set(title='PR Curves — All Baselines', xlabel='Recall', ylabel='Precision')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('../results/figures/exp1_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save leaderboard
leaderboard.to_csv('../results/exp1_baseline_leaderboard.csv')
print('Saved leaderboard to results/exp1_baseline_leaderboard.csv')